# GenAI-Net (RL4CRN) Tutorial 06 — Robust SSA Task (Smoke + Wiring)

Compact tutorial for the **robust SSA** task.

Robust SSA typically includes additional penalty/robustness terms (e.g., mean tracking + variability control).
This notebook focuses on task construction, wiring, and a reward smoke test.

---
## 0) Environment sanity check


In [ ]:
import os, sys, numpy as np

print("Python:", sys.version.split()[0])
print("CWD:", os.getcwd())


---
## 1) Imports

We use the standard interface layer:
- `Configurator`, `make_task`, `make_session_and_trainer`


In [ ]:
import numpy as np

from RL4CRN.utils.input_interface import (
    Configurator,
    make_task,
    make_session_and_trainer,
)


---
## 2) Notebook-local helpers


In [ ]:
def print_task_summary(task, max_preview=3):
    print("Task:", task.name)
    print("time_horizon:", task.time_horizon.shape, f"[0..{task.time_horizon[-1]}]")
    print("num scenarios:", len(task.u_list))
    if len(task.u_list) > 0:
        print(f"first {min(max_preview, len(task.u_list))} u:", task.u_list[:max_preview])
    print()

def run_smoke_reward(task, state, label=""):
    out = task.compute_reward(state)
    if isinstance(out, tuple):
        loss, info = out
    else:
        loss, info = out, {}
    print(f"[reward smoke{(' - ' + label) if label else ''}] loss={float(loss):.6g} | info_keys={list(info.keys())[:10]}")
    return out


---
## 3) Define the robust SSA task (standalone)


In [ ]:
species_labels = ["X_1","X_2","X_3","X_4","X_5","X_6","OUT"]

task = make_task(
    kind="ssa_robust",
    species_labels=species_labels,
    p=2,
    u_values=[1.0, 2.0, 3.0],
    target="copy_input0",
    ic="zero",
    weights="steady_state",
    t_f=30, n_t=80,
    n_trajectories=64,
    max_threads=1024,
    rpa_weight=3.0,
    cv_weight=1.0,
)

print_task_summary(task)


---
## 4) Full wiring + reward smoke test


In [ ]:
cfg = Configurator.preset("fast")

# ---- Task ----
cfg.task.kind = "ssa_robust"
cfg.task.n_inputs = 2
cfg.task.p = 2
cfg.task.u_values = [1.0, 2.0, 3.0]
cfg.task.target = "copy_input0"
cfg.task.weights = "steady_state"
cfg.task.t_f = 30.0
cfg.task.N_t = 80
cfg.task.ic_value = 0.0

# Robustness weights (if exposed via cfg)
# cfg.task.rpa_weight = 3.0
# cfg.task.cv_weight = 1.0

session, trainer = make_session_and_trainer(cfg, device="auto")
print_task_summary(session.task)

run_smoke_reward(session.task, session.crn_template, label="ssa robust template")


---
## 5) (Optional) Training

Start small and scale carefully; robust terms can increase variance and runtime in SSA.


In [ ]:
# Uncomment to try training (may be expensive):
# cfg.train.max_added_reactions = 3
# cfg.train.epochs = 3
# cfg.train.render_every = 1
# cfg.train.seed = 5
# trainer.run(epochs=cfg.train.epochs, checkpoint_path=None)
# trainer.inspect_best(plot=True)
